In [37]:
import pandas as pd

test_df = pd.read_csv('../data/processed_data/test.csv')
schedules_df = pd.read_csv('../data/original_data/schedules_to_may_2024.csv', sep='|')

test_df.head()

,ID,vesselId,time,scaling_factor,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,day_of_the_week,hour_of_the_day,...,minute_sin,minute_cos,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day
0,4,61e9f38eb937134a3c4bfd8d,0.349750,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,0.975528,0.654508,1,1,0,0,0,0,0,0
1,201,61e9f38eb937134a3c4bfd8d,0.349802,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,0.095492,0.206107,1,1,1,0,0,0,0,0
2,583,61e9f38eb937134a3c4bfd8d,0.349904,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,0.345492,0.024472,1,0,0,0,0,0,0,0
3,701,61e9f38eb937134a3c4bfd8d,0.349938,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,0.095492,0.793893,1,0,0,0,0,0,0,0
4,829,61e9f38eb937134a3c4bfd8d,0.349961,0.3,0.363636,0.346154,0.350685,0.233333,0.333333,0.086957,...,0.654508,0.975528,1,1,0,0,0,0,0,0


In [38]:
schedules_df.head()

,vesselId,shippingLineId,shippingLineName,arrivalDate,sailingDate,portName,portId,portLatitude,portLongitude
0,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-10-02 00:00:00+00:00,2023-10-03 00:00:00+00:00,Port of Brunswick,61d38499b7b7526e1adf3d54,31.140556,-81.496667
1,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-10-27 00:00:00+00:00,2023-10-27 00:00:00+00:00,Port of Southampton,61d3832bb7b7526e1adf3b63,50.902500,-1.428889
2,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-10-19 00:00:00+00:00,2023-10-20 00:00:00+00:00,Port of Bremerhaven,61d375e793c6feb83e5eb3e2,53.563611,8.554722
3,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-10-09 00:00:00+00:00,2023-10-10 00:00:00+00:00,Port of New York,61d38481b7b7526e1adf3d23,40.688333,-74.028611
4,61e9f3b1b937134a3c4bfe53,61a8e672f9cba188601e84ac,Wallenius Wilhelmsen Ocean,2023-09-25 00:00:00+00:00,2023-09-26 00:00:00+00:00,Manzanillo International Terminal,61d37d0199db2ccf7339eee1,9.372370,-79.879790


In [ ]:
import pandas as pd
import numpy as np

def engineer_port_locations(schedule_df, test_df):
    """
    Engineer port latitude and longitude features for test data based on vessel schedules.
    Handles timezone-aware datetime comparisons.
    
    Parameters:
    schedule_df (pd.DataFrame): Schedule data with vessel ports and times
    test_df (pd.DataFrame): Test data requiring port location features
    
    Returns:
    pd.DataFrame: Test data with added port_lat and port_long features
    """
    # Convert string data to datetime and ensure timezone consistency
    schedule_df = schedule_df.copy()
    test_df = test_df.copy()
    
    # Convert schedule dates to UTC
    schedule_df['arrivalDate'] = pd.to_datetime(schedule_df['arrivalDate']).dt.tz_convert('UTC')
    schedule_df['sailingDate'] = pd.to_datetime(schedule_df['sailingDate']).dt.tz_convert('UTC')
    
    # Convert test dates to UTC
    test_df['time_utc'] = pd.to_datetime(test_df['time']).dt.tz_localize('UTC')
    
    # Initialize new columns
    test_df['port_lat'] = np.nan
    test_df['port_long'] = np.nan
    
    # Process each row in test_df
    for idx, row in test_df.iterrows():
        vessel_schedule = schedule_df[schedule_df['vesselId'] == row['vesselId']].copy()
        
        if len(vessel_schedule) == 0:
            continue
            
        # Sort vessel schedule by arrival date
        vessel_schedule = vessel_schedule.sort_values('arrivalDate')
        
        # Find the relevant port
        # Case 1: Vessel is at a port (between arrival and sailing)
        at_port = vessel_schedule[
            (vessel_schedule['arrivalDate'] <= row['time_utc']) & 
            (vessel_schedule['sailingDate'] >= row['time_utc'])
        ]
        
        if len(at_port) > 0:
            # Use the current port's coordinates
            test_df.at[idx, 'port_lat'] = at_port.iloc[0]['portLatitude']
            test_df.at[idx, 'port_long'] = at_port.iloc[0]['portLongitude']
            continue
        
        # Case 2: Vessel is between ports
        next_port = vessel_schedule[vessel_schedule['arrivalDate'] > row['time_utc']].iloc[0] if len(vessel_schedule[vessel_schedule['arrivalDate'] > row['time_utc']]) > 0 else None
        prev_port = vessel_schedule[vessel_schedule['sailingDate'] < row['time_utc']].iloc[-1] if len(vessel_schedule[vessel_schedule['sailingDate'] < row['time_utc']]) > 0 else None
        
        if prev_port is not None and next_port is not None:
            # Calculate time ratios
            total_time = (next_port['arrivalDate'] - prev_port['sailingDate']).total_seconds()
            elapsed_time = (row['time_utc'] - prev_port['sailingDate']).total_seconds()
            ratio = elapsed_time / total_time
            
            # Interpolate coordinates
            test_df.at[idx, 'port_lat'] = prev_port['portLatitude'] + (next_port['portLatitude'] - prev_port['portLatitude']) * ratio
            test_df.at[idx, 'port_long'] = prev_port['portLongitude'] + (next_port['portLongitude'] - prev_port['portLongitude']) * ratio
        
        elif prev_port is not None:
            # Use last known port
            test_df.at[idx, 'port_lat'] = prev_port['portLatitude']
            test_df.at[idx, 'port_long'] = prev_port['portLongitude']
        
        elif next_port is not None:
            # Use next port
            test_df.at[idx, 'port_lat'] = next_port['portLatitude']
            test_df.at[idx, 'port_long'] = next_port['portLongitude']

    test_df.drop(["time_utc"], axis=1, inplace=True)
    
    return test_df

test_df = engineer_port_locations(test_df=test_df, schedule_df=schedules_df)



In [40]:
test_df.head(50)

,ID,vesselId,time,scaling_factor,month_of_the_year,week_of_the_year,day_of_the_year,day_of_the_month,day_of_the_week,hour_of_the_day,...,time_diff_gt_10min,time_diff_gt_20min,time_diff_gt_40min,time_diff_gt_1hour,time_diff_gt_2hours,time_diff_gt_6hours,time_diff_gt_12hours,time_diff_gt_1day,port_lat,port_long
0,4,61e9f38eb937134a3c4bfd8d,0.349750,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,1,1,0,0,0,0,0,0,NaN,NaN
1,201,61e9f38eb937134a3c4bfd8d,0.349802,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.000000,...,1,1,1,0,0,0,0,0,NaN,NaN
2,583,61e9f38eb937134a3c4bfd8d,0.349904,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,1,0,0,0,0,0,0,0,NaN,NaN
3,701,61e9f38eb937134a3c4bfd8d,0.349938,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.043478,...,1,0,0,0,0,0,0,0,NaN,NaN
4,829,61e9f38eb937134a3c4bfd8d,0.349961,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.086957,...,1,1,0,0,0,0,0,0,NaN,NaN
5,1038,61e9f38eb937134a3c4bfd8d,0.350024,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.086957,...,1,0,0,0,0,0,0,0,NaN,NaN
6,1114,61e9f38eb937134a3c4bfd8d,0.350052,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.086957,...,1,1,0,0,0,0,0,0,NaN,NaN
7,1258,61e9f38eb937134a3c4bfd8d,0.350092,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.130435,...,0,0,0,0,0,0,0,0,NaN,NaN
8,1396,61e9f38eb937134a3c4bfd8d,0.350109,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.130435,...,1,1,0,0,0,0,0,0,NaN,NaN
9,1540,61e9f38eb937134a3c4bfd8d,0.350166,0.30,0.363636,0.346154,0.350685,0.233333,0.333333,0.130435,...,1,1,0,0,0,0,0,0,NaN,NaN


In [41]:
test_df.to_csv("../data/processed_data/test.csv", index=False)